# 提前还贷分析模型

评估提前还贷对现金流的影响，对比缩短年限和减少月供两种策略，
并与理财投资收益进行对比决策。

**模型公式：**
```
月供 = 贷款余额 × r × (1+r)^n / ((1+r)^n - 1)  （等额本息）
省息 = 原剩余利息 - 新方案剩余利息
```


---
## 一、输入参数

修改以下参数后从头运行即可刷新全部输出。


In [ ]:
# ============================================================
# 贷款原始参数
# ============================================================

loan_balance = 1_200_000       # 当前贷款余额（元）
loan_rate = 0.026              # 年利率（如 2.6%）
remaining_years = 30           # 剩余还款年限
repayment_method = "等额本息"  # 还款方式：等额本息

# ============================================================
# 提前还款方案
# ============================================================

prepay_amount = 300_000        # 计划提前还款金额（元）
prepay_strategy = "缩短年限"   # 策略：缩短年限 / 减少月供

# ============================================================
# 对比参数
# ============================================================

investment_return = 0.04       # 预期理财年化收益率（如 4%）

# ============================================================
# 计算参数
# ============================================================

monthly_rate = loan_rate / 12
total_months = remaining_years * 12

print(f"当前贷款余额: {loan_balance:,.0f} 元")
print(f"年利率: {loan_rate*100:.2f}%")
print(f"剩余还款: {remaining_years} 年 ({total_months} 期)")
print(f"提前还款金额: {prepay_amount:,.0f} 元")
print(f"策略: {prepay_strategy}")


---
## 二、原贷款计划

当前贷款在不提前还款情况下的还款计划。


In [ ]:
# ============================================================
# 等额本息计算函数
# ============================================================


def calc_monthly_payment(principal, monthly_rate, n_months):
    """计算月供"""
    if monthly_rate == 0:
        return principal / n_months
    return principal * monthly_rate * (1 + monthly_rate)**n_months / ((1 + monthly_rate)**n_months - 1)


def calc_total_interest(principal, monthly_rate, n_months):
    """计算总利息"""
    monthly = calc_monthly_payment(principal, monthly_rate, n_months)
    return monthly * n_months - principal


# ============================================================
# 原贷款计划
# ============================================================

orig_monthly = calc_monthly_payment(loan_balance, monthly_rate, total_months)
orig_total_interest = calc_total_interest(loan_balance, monthly_rate, total_months)
orig_total_payment = loan_balance + orig_total_interest

print("=" * 60)
print("  原贷款计划")
print("=" * 60)
print(f"\n月供: {orig_monthly:>12,.2f} 元")
print(f"剩余利息: {orig_total_interest:>10,.2f} 元")
print(f"本息合计: {orig_total_payment:>10,.2f} 元")
print(f"剩余期数: {total_months:>10} 期")


---
## 三、提前还贷方案

对比两种策略：**缩短年限**（月供不变，更快还清）vs **减少月供**（年限不变，每月压力减小）。


In [ ]:
# ============================================================
# 提前还贷计算
# ============================================================

import math

new_balance = loan_balance - prepay_amount  # 还款后剩余本金

print("=" * 65)
print("  提前还贷方案")
print("=" * 65)
print(f"\n提前还款: {prepay_amount:>12,.0f} 元")
print(f"剩余本金: {new_balance:>12,.0f} 元")

# ----- 方案A：缩短年限（月供不变）-----
print("\n" + "-" * 50)
print("  方案A：缩短年限（月供不变）")
print("-" * 50)

if monthly_rate > 0:
    if orig_monthly > new_balance * monthly_rate:
        n_new_a = math.log(orig_monthly / (orig_monthly - new_balance * monthly_rate)) / math.log(1 + monthly_rate)
        n_new_a = math.ceil(n_new_a)
    else:
        n_new_a = 1
else:
    n_new_a = math.ceil(new_balance / orig_monthly)

if monthly_rate > 0:
    remaining_a = (new_balance * (1 + monthly_rate) ** n_new_a -
                   orig_monthly * ((1 + monthly_rate) ** n_new_a - 1) / monthly_rate)
    if remaining_a < 0:
        remaining_a = 0
else:
    remaining_a = 0

years_new_a = n_new_a // 12
months_new_a = n_new_a % 12

new_interest_a = orig_monthly * (n_new_a - 1) + remaining_a - new_balance
interest_saved_a = max(0, orig_total_interest - new_interest_a)

print(f"  月供不变: {orig_monthly:>10,.2f} 元")
print(f"  剩余期数: {n_new_a} 期（{years_new_a} 年 {months_new_a} 个月）")
print(f"  节省年限: {total_months - n_new_a} 期")
print(f"  节省利息: {interest_saved_a:>10,.2f} 元")

# ----- 方案B：减少月供（年限不变）-----
print("\n" + "-" * 50)
print("  方案B：减少月供（年限不变）")
print("-" * 50)

new_monthly_b = calc_monthly_payment(new_balance, monthly_rate, total_months)
new_interest_b = calc_total_interest(new_balance, monthly_rate, total_months)
interest_saved_b = orig_total_interest - new_interest_b

print(f"  新月供: {new_monthly_b:>12,.2f} 元")
print(f"  月供减少: {orig_monthly - new_monthly_b:>10,.2f} 元")
print(f"  节省利息: {interest_saved_b:>10,.2f} 元")

# ----- 推荐 -----
print("\n" + "=" * 50)
print("  推荐")
print("=" * 50)
if interest_saved_a > interest_saved_b:
    print(f"  方案A（缩短年限）省息更多，多省 {interest_saved_a - interest_saved_b:,.0f} 元")
elif interest_saved_a < interest_saved_b:
    print(f"  方案B（减少月供）省息更多，多省 {interest_saved_b - interest_saved_a:,.0f} 元")
else:
    print("  两种方案省息金额相同")


---
## 四、还贷 vs 理财

提前还贷相当于获得了一个无风险收益率（即贷款利率），对比将资金用于理财的预期收益。

**决策逻辑：**
- 贷款利率 > 理财收益率 → 提前还贷更划算
- 贷款利率 < 理财收益率 → 理财更划算


In [ ]:
# ============================================================
# 还贷 vs 理财对比
# ============================================================

print("=" * 55)
print("  还贷 vs 理财对比")
print("=" * 55)

effective_return = loan_rate * 100
invest_return_pct = investment_return * 100

print(f"\n提前还贷等效年化收益率: {effective_return:.2f}%（即贷款利率，无风险）")
print(f"理财产品预期年化收益率:  {invest_return_pct:.2f}%（有风险）")

print("\n" + "-" * 40)
if loan_rate > investment_return:
    gap = (loan_rate - investment_return) * 100
    print("  结论：提前还贷更划算")
    print(f"        还贷等效收益高出理财 {gap:.2f} 个百分点")
elif loan_rate < investment_return:
    gap = (investment_return - loan_rate) * 100
    print("  结论：理财更划算")
    print(f"        理财预期收益高出还贷 {gap:.2f} 个百分点，但需注意风险")
else:
    print("  结论：两者基本持平")

print("\n" + "-" * 40)
print("  省息与理财收益对比（至原贷款到期）：")

invest_earning = prepay_amount * ((1 + investment_return) ** remaining_years - 1)

print(f"\n  提前还贷省息:      {interest_saved_a:>10,.2f} 元（方案A）")
print(f"                      {interest_saved_b:>10,.2f} 元（方案B）")
print(f"  {prepay_amount:,}元理财{remaining_years}年收益: {invest_earning:>10,.2f} 元（预期）")


---
## 五、可视化

原计划 vs 新方案的还款情况对比。


In [ ]:
# ============================================================
# 可视化 — 月供与利息对比
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams["font.sans-serif"] = ["WenQuanYi Micro Hei", "Noto Sans CJK SC",
                                    "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ======== 图1: 月供对比 ========
labels = ["原计划", "缩短年限", "减少月供"]
monthly_vals = [orig_monthly, orig_monthly, new_monthly_b]
colors_m = ["#3498db", "#e74c3c", "#2ecc71"]

bars1 = ax1.bar(labels, monthly_vals, color=colors_m, alpha=0.8, width=0.5)
for bar, val in zip(bars1, monthly_vals):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
             f"{val:,.0f}", ha="center", va="bottom", fontsize=10)
ax1.set_ylabel("月供（元）")
ax1.set_title("月供对比")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# ======== 图2: 总利息对比 ========
interest_vals = [orig_total_interest, new_interest_a, new_interest_b]
bars2 = ax2.bar(labels, interest_vals, color=colors_m, alpha=0.8, width=0.5)
for bar, val in zip(bars2, interest_vals):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2000,
             f"{val:,.0f}", ha="center", va="bottom", fontsize=10)
ax2.set_ylabel("利息总额（元）")
ax2.set_title("剩余利息对比")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.show()


---
## 六、生成报告

将分析结果输出为 Markdown 报告和/或 PNG 图片。


In [ ]:
# ============================================================
# 生成分析报告 — Markdown 和/或 PNG 图片输出
# ============================================================

report_format = "both"

import datetime
import os
import markdown as md_lib
import weasyprint
import pymupdf
from PIL import Image as PILImage
from IPython.display import display, Image

today = datetime.date.today().strftime("%Y-%m-%d")
report_dir = "reports"
os.makedirs(report_dir, exist_ok=True)

# ----- 构建报告内容 -----
report  = f"# 提前还贷分析报告\n\n"
report += f"**生成日期：** {today}\n\n"
report += f"---\n\n"
report += f"## 一、输入参数\n\n"
report += f"### 贷款信息\n\n"
report += f"| 参数 | 值 |\n"
report += f"|---|---|\n"
report += f"| 贷款余额 | {loan_balance:,} 元 |\n"
report += f"| 年利率 | {loan_rate*100:.2f}% |\n"
report += f"| 剩余年限 | {remaining_years} 年 |\n"
report += f"| 还款方式 | {repayment_method} |\n"
report += f"| 月供 | {orig_monthly:,.2f} 元 |\n"
report += f"| 剩余利息 | {orig_total_interest:,.2f} 元 |\n"
report += f"\n"
report += f"### 提前还款方案\n\n"
report += f"| 参数 | 值 |\n"
report += f"|---|---|\n"
report += f"| 提前还款金额 | {prepay_amount:,} 元 |\n"
report += f"| 策略 | {prepay_strategy} |\n"
report += f"\n---\n\n"
report += f"## 二、方案A：缩短年限（月供不变）\n\n"
report += f"| 指标 | 值 |\n"
report += f"|---|---|\n"
report += f"| 提前还款后本金 | {new_balance:,} 元 |\n"
report += f"| 月供 | {orig_monthly:,.2f} 元 |\n"
report += f"| 剩余期数 | {n_new_a} 期（{years_new_a} 年 {months_new_a} 个月） |\n"
report += f"| 节省利息 | {interest_saved_a:,.2f} 元 |\n"
report += f"\n"
report += f"## 三、方案B：减少月供（年限不变）\n\n"
report += f"| 指标 | 值 |\n"
report += f"|---|---|\n"
report += f"| 提前还款后本金 | {new_balance:,} 元 |\n"
report += f"| 新月供 | {new_monthly_b:,.2f} 元 |\n"
report += f"| 月供减少 | {orig_monthly - new_monthly_b:,.2f} 元 |\n"
report += f"| 节省利息 | {interest_saved_b:,.2f} 元 |\n"
report += f"\n---\n\n"
report += f"## 四、还贷 vs 理财\n\n"
report += f"| 项目 | 收益率 |\n"
report += f"|---|---|\n"
report += f"| 提前还贷等效收益率 | {loan_rate*100:.2f}%（无风险） |\n"
report += f"| 理财预期收益率 | {investment_return*100:.2f}%（有风险） |\n"
report += f"\n"
if loan_rate > investment_return:
    report += "**结论：提前还贷更划算。** 还贷等效收益高于理财预期。\n"
elif loan_rate < investment_return:
    report += "**结论：理财更划算。** 理财预期收益高于还贷等效收益，但需注意市场风险。\n"
else:
    report += "**结论：两者基本持平。**\n"
report += "\n---\n\n"
report += "## 五、方法说明\n\n"
report += "**模型：** 等额本息提前还贷分析模型。\n\n"
report += "**计算公式：**\n"
report += "```\n"
report += "月供 = P × r × (1+r)^n / ((1+r)^n - 1)\n"
report += "```\n\n"
report += "**策略说明：**\n"
report += "- **缩短年限：** 月供不变，剩余还款期数减少，省息最多\n"
report += "- **减少月供：** 还款年限不变，月供降低，现金流压力减小\n"
report += "\n"
report += "> **免责声明：** 本报告仅为基于给定参数的财务分析工具，不构成投资建议。\n"

# ----- 保存 Markdown -----
if report_format in ("markdown", "both"):
    md_file = f"{report_dir}/{today}-提前还贷分析报告.md"
    with open(md_file, "w", encoding="utf-8") as f:
        f.write(report)
    print(f"Markdown 报告: {md_file}")

# ----- 生成图片 -----
if report_format in ("image", "both"):
    html_body = md_lib.markdown(report, extensions=["tables", "fenced_code"])

    html_template = f"""<!DOCTYPE html>\n<html>\n<head>\n<meta charset="utf-8">\n<style>\n  @page {{ size: A4; margin: 1.8cm; }}\n  body {{ font-family: "WenQuanYi Micro Hei", "Noto Sans CJK SC", sans-serif;\n          font-size: 11pt; line-height: 1.7; color: #333; }}\n  h1 {{ font-size: 20pt; text-align: center; color: #1a1a1a; margin-bottom: 4pt; }}\n  h2 {{ font-size: 16pt; color: #2c3e50; border-bottom: 2px solid #3498db;\n        padding-bottom: 3pt; margin-top: 18pt; }}\n  table {{ border-collapse: collapse; width: 100%; margin: 8pt 0; font-size: 10pt; }}\n  th, td {{ border: 1px solid #ccc; padding: 5pt 8pt; text-align: center; }}\n  th {{ background-color: #3498db; color: white; font-weight: bold; }}\n  tr:nth-child(even) {{ background-color: #f8f9fa; }}\n  strong {{ color: #2c3e50; }}\n  blockquote {{ border-left: 4px solid #e74c3c; margin: 10pt 0; padding: 6pt 12pt;\n               background-color: #fdf2f2; color: #666; font-size: 10pt; }}\n  hr {{ border: none; border-top: 1px solid #eee; margin: 12pt 0; }}\n  p {{ margin: 6pt 0; }}\n</style>\n</head>\n<body>\n{html_body}\n</body>\n</html>"""

    img_file = f"{report_dir}/{today}-提前还贷分析报告.png"
    pdf_data = weasyprint.HTML(string=html_template).write_pdf()
    doc = pymupdf.open("pdf", pdf_data)
    pages = [doc[i].get_pixmap(dpi=200) for i in range(doc.page_count)]
    total_h = sum(p.height for p in pages)
    canvas = PILImage.new("RGB", (pages[0].width, total_h))
    y = 0
    for p in pages:
        img = PILImage.frombytes("RGB", (p.width, p.height), p.samples)
        canvas.paste(img, (0, y))
        y += p.height
    canvas.save(img_file)
    num_pages = doc.page_count
    doc.close()
    print(f"图片报告: {img_file} ({num_pages} 页拼接)")

    display(Image(filename=img_file))
